[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/solar_cell.ipynb)

# A Solar Cell's Fingers

This example computes two-dimensional carrier transport across one repeating period of a silicon solar cell between the front-contact metal fingers and finds the finger spacing that maximizes delivered power. Light entering between the opaque fingers decays with depth and generates electron-hole pairs throughout the donor-doped base. Holes diffuse upward to the shallow acceptor-doped emitter layer, cross the p–n junction, and flow laterally along that thin layer to the nearest finger while electrons flow down to the rear metal contact. Widening the gap between fingers admits more light by shading a smaller fraction of the front surface, while forcing holes to travel further along the resistive emitter and lose more voltage on the way, so delivered power peaks at an intermediate spacing.

Planar spacetime across the repeating cell period is weighed by a null point weight `w` and joined with null quantity directions for electrons, holes, heat, and light, where the term ***parcel*** here means an extensive state combining carrier counts, charge along `time`, heat, or photon energy. Pairing `y & vertices` reads depth below the front surface for the exponential optical generation profile outside the finger shadow, wedging the potential plane with the spatial gradient forms one combined drift-diffusion drive `-(gradient ^ potentials)`, and `drive.commutator(hole)` gives the planar hole flow field streaked across the cross-section. Pulling fluxes back through `(gradient ^ Potentials).adjugate()` sums contributions only from incident triangles, enforcing mirror symmetry along the period boundaries with no extra boundary terms; see also [the 1D p–n junction](diode.ipynb) and [the planar MOSFET](mosfet.ipynb).

In vector-calculus drift–diffusion notation, two-dimensional continuity $\nabla\cdot\mathbf J_n = q(R - G)$ and $-\nabla\cdot\mathbf J_p = q(R - G)$ with $\mathbf J_n = -q\mu_n n\nabla\psi + q D_n\nabla n$, $\mathbf J_p = -q\mu_p p\nabla\psi - q D_p\nabla p$, and $-\nabla\cdot(\epsilon\nabla\psi) = q(p - n + N_D^+ - N_A^-)$ requires explicit homogeneous Neumann conditions $\hat{\mathbf n}\cdot\nabla\psi = \hat{\mathbf n}\cdot\mathbf J_n = \hat{\mathbf n}\cdot\mathbf J_p = 0$ on the symmetry planes and insulated front gap, reducing in the one-dimensional emitter-sheet approximation to the lumped fractional power loss $\rho_\square J_{\mathrm{mp}} S^2 / (12 V_{\mathrm{mp}}) + w / S$ minimized at $S^3 = 6 w V_{\mathrm{mp}} / (\rho_\square J_{\mathrm{mp}})$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import partial

import numpy as np

from examples.thermodynamics.semiconductor_devices import render, scenarios
from examples.util.animation import show_states
from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag

# Planar projective spacetime weighed by w0, joined with null quantity directions for electrons, holes, heat, and light.
ga = Algebra("x+y+t-w0") * Algebra("n0p0q0l0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
Upright = Plane.derive.within(Algebra("x+y+"))                                # Vector+
Potentials = Plane.derive.within(Algebra("t-n0p0"))                           # Vector+
Drive = ga.gatype(Upright ^ Potentials)                                       # Bivector+
# Unit charge along time, charged carrier parcels, and neutral heat and photon energy parcels.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
heat = mv.q.dual()                                                            # [] Point-
light = mv.l.dual()                                                           # [] Point-
pairs = electron + hole                                                       # [] Point-
carriers = stack((electron, hole))                                            # [2] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [2] Point-
count_planes = stack((mv.n, mv.p))                                            # [2] Plane+
charges = carriers - counts                                                   # [2] Point-
# Span of time and every material quantity along which purely spatial fields stay constant.
unread = time & counts[0] & counts[1] & heat & light                          # [] Span-

## Repeating cell period

One repeating period extends horizontally from the midpoint of one front finger to the midpoint of the next, with mirror symmetry at both vertical boundaries. A donor-doped base 160 Debye lengths deep lies beneath an acceptor-doped emitter 8 Debye lengths deep and above a heavily donor-doped back-surface field layer that repels minority holes from the rear metal contact, while opaque fingers 16 Debye lengths wide contact the front emitter. Graded horizontal rows resolve the shallow emitter junction and rear barrier identically at every spacing while cosine-clustered columns stretch smoothly across the illuminated gap.

In [ ]:
intrinsic, gap = 1e-6, 43.6
mobilities, permittivity = np.array([1.0, 1 / 3]), 1.0
lifetime = 2e5
rate_constant = 1 / lifetime
surface_generation, absorption_length = 1.5e-4, 50.0

In [ ]:
finger_spacings = scenarios.finger_spacings
n_spacings = len(finger_spacings)
topology, coordinates, doping, shadow_mask, contacts = scenarios.period_grid(finger_spacings[0])
V, E, F = topology.n_elements
vertices = (mv(Upright, coordinates) + mv.w).dual().field()                   # Point-[V]
dopants = (time * doping).field()                                             # Point-[V]
fingers, back = contacts
held_indices, free_indices = scenarios.contact_partition(contacts, V)
held_reading = topology.reading(held_indices, 0)                              # Scalar+[V_held <- V]
free_reading = topology.reading(free_indices, 0)                              # Scalar+[V_free <- V]
depth_profile = surface_generation * ((mv.y & vertices) / absorption_length).exp()   # Scalar+[V]
generation = mv.scalar(np.where(shadow_mask, 0.0, 1.0)[:, None]).field() * depth_profile   # Scalar+[V]
render.cell_figure(topology, vertices, dopants, contacts, generation, finger_spacings[0]);

At the narrowest finger spacing, tiled periods show dark front metal fingers above the rose emitter band, warm yellow optical generation decaying downward through the base, white unlit columns beneath each finger shadow, and the blue donor-rich back-surface layer above the rear contact.

## Coupled drift, diffusion, and electrostatics

The term ***simplex gradient*** here means the spatial covector at each corner of a triangle whose pairing with a planar displacement gives the linear change of that corner's interpolation weight across the triangle. Joining each opposite triangle side with `unread`, dividing by its pairing with the corner, and projecting onto `Upright` yields the corner simplex gradients, rebuilt from `vertices` alone whenever the finger spacing stretches the grid. Pulling fluxes back through `(gradient ^ Potentials).adjugate()` satisfies `potentials & outflow(flux) == drive_map(potentials) & flux` and gathers incident triangle fluxes into dual-cell outflows.

In steady state, dual-cell carrier outflows balance mass-action recombination and optical pair generation while Gauss's law supplies the electrostatic charge balance along `time`, solved at each bias by density-weighted damped Newton iteration.

In [ ]:
corner_mean = topology.incidence[2, 0].mean(axis=0)                           # Scalar+[F <- V]


def simplex_maps(current_vertices: Point) -> tuple[Scalar, Scalar, SparseExtensor]:
    """Triangle corner joins at `current_vertices` (`Point-[V]`) determine the triangle areas (`Scalar+[F]`), lumped
    dual areas (`Scalar+[V]`), and planar simplex gradient operator (`Upright+[F <- V]`)."""
    corners = topology.incidence[2, 0] * current_vertices                     # [3] Point-[F]
    facing = topology.off[2, 0] * current_vertices                            # [2, 3] Point-[F]
    sides = facing[0] & facing[1]                                             # [3] Line-[F]
    gradients = ((sides & unread) / (sides & unread & corners)).cast(Upright)   # [3] Upright+[F]
    areas = (corners[0] & corners[1] & corners[2] & unread).norm() / 2        # Scalar+[F]
    gradient = (spdiag(gradients) * topology.incidence[2, 0]).sum(axis=0)     # Upright+[F <- V]
    return areas, corner_mean.adjugate() * areas, gradient

In [ ]:
unit_conductances = mobilities * Drive.commutator(carriers).regressive(carriers)   # [2] Flux- <- Drive+
displacement = permittivity * Drive.commutator(time).regressive(time)         # [] Flux- <- Drive+


def carrier_densities(potentials: Potentials) -> Scalar:
    """Boltzmann's law relates the electron and hole densities (`[2] Scalar+[V]`) to the chemical potentials."""
    return intrinsic * (potentials & counts).exp()


def steady_states(areas: Scalar, dual_areas: Scalar, gradient: SparseExtensor, potentials: Potentials,
                  contacts: Potentials, generations: Scalar) -> Iterator[Potentials]:
    """Yields steady-state vertex potentials (`Potentials+[V]`) across held `contacts` (`[n_frames] Potentials+[V_held]`)
    and optical `generations` (`[n_frames] Scalar+[V]`)."""
    drive_map = -(gradient ^ Potentials)                                      # (Drive+ <- Potentials+)[F <- V]
    outflow = drive_map.adjugate()                                            # (Point- <- Flux-)[V <- F]
    for contact, frame_generation in zip(contacts, generations):
        potentials = free_reading.adjugate() * (free_reading * potentials) + held_reading.adjugate() * contact
        while True:
            densities = carrier_densities(potentials)                         # [2] Scalar+[V]
            drive = drive_map(potentials)                                     # Drive+[F]
            conductance = ((corner_mean * densities) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[F]
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2) - frame_generation   # Scalar+[V]
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[V]
            gauss = outflow(areas * displacement(drive)) - dual_areas * charge   # Point-[V]
            residual = outflow(areas * conductance(drive)) + dual_areas * rates * pairs + gauss   # Point-[V]
            capacitance = dual_areas * (densities * -charges * (Potentials & counts)).sum(axis=0)
            drive_response = (areas * (conductance + displacement))(drive_map)
            density_response = (spdiag(areas * unit_conductances(drive)) * corner_mean
                                * spdiag(densities * (Potentials & counts))).sum(axis=0)
            pair_response = dual_areas * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs))
            jacobian = capacitance + pair_response + outflow(drive_response + density_response)
            balance_weights = time * (mv.t & Point) + (counts * (count_planes & Point) / densities).sum(axis=0)
            free_jacobian = free_reading * balance_weights(jacobian) * free_reading.adjugate()
            step_change = -(free_reading.adjugate() * free_jacobian.solve(free_reading * balance_weights(residual)))
            chemical_shift = (counts & step_change).abs().sum(axis=0)         # Scalar+[V]
            potentials = potentials + step_change / (1 + chemical_shift / limit)
            if chemical_shift.max() < tolerance:
                break
        yield potentials

## Illumination and open-circuit voltage

Ohmic contacts hold both the front fingers and the rear contact at local charge neutrality while raising the front-finger electrostatic potential by the terminal forward bias. Starting in the dark at short circuit, illumination rises to full intensity at zero voltage before forward bias sweeps with finer steps near open circuit, reading the delivered current leaving the rear contact via `t & outflow(areas * flux)`. Running the cells below takes about fifteen seconds.

In [ ]:
limit, tolerance = 6.0, 0.1
frame_duration_ms = 40
biases, shares = scenarios.lighting_biases()
n_frames = len(biases)
bias_extensor = mv.scalar(biases[:, None])                                    # [n_frames] Scalar+
volt = 1 / 0.02585

In [ ]:
chemical = scenarios.neutral_potentials(doping)                               # [2, V]
neutral = (mv.t * chemical[0] + mv.n * chemical[0] + mv.p * chemical[1]).field()   # Potentials+[V]


def terminal_readings(areas: Scalar, gradient: SparseExtensor, potentials: Potentials) -> tuple[Scalar, Scalar]:
    """Finger-to-back quasi-Fermi drop and rear charge outflow determine the cell voltage and delivered current
    (`[] Scalar+` each)."""
    drive_map = -(gradient ^ Potentials)                                      # (Drive+ <- Potentials+)[F <- V]
    levels = potentials & electron                                            # Scalar+[V]
    rise = (topology.reading(back, 0) * levels).sites.mean() - (topology.reading(fingers, 0) * levels).sites.mean()
    conductance = ((corner_mean * carrier_densities(potentials)) * unit_conductances).sum(axis=0)   # (Flux- <- Drive+)[F]
    net_outflow = drive_map.adjugate()(areas * conductance(drive_map(potentials)))   # Point-[V]
    return rise, (topology.reading(back, 0) * (mv.t & net_outflow)).sites.sum()


def hole_flows(gradient: SparseExtensor, potentials: Potentials) -> Point:
    """The planar drive `-(gradient ^ potentials)` sweeps holes across each triangle into their flow field
    (`Point-[F]`)."""
    densities = corner_mean * carrier_densities(potentials)[1]                # Scalar+[F]
    return mobilities[1] * densities * (-(gradient ^ potentials)).commutator(hole)


areas, dual_areas, gradient = simplex_maps(vertices)
reference_flow = surface_generation * absorption_length
generated_current = (dual_areas * generation).sites.sum()                     # [] Scalar+
finger_mask = np.concatenate([np.ones(len(fingers)), np.zeros(len(back))])
voltages = bias_extensor[:, None] * finger_mask                               # [n_frames, V_held] Scalar+
held_potentials = held_reading * neutral + (mv.t * voltages).field()          # [n_frames] Potentials+[V_held]
lit_states = show_states(
    steady_states(areas, dual_areas, gradient, neutral, held_potentials, shares * generation),
    render.lighting_animation, topology, vertices, dopants, contacts, generation,
    partial(hole_flows, gradient), partial(terminal_readings, areas, gradient), reference_flow,
    finger_spacings[0], biases.max(), generated_current, volt, shares, name="solar_cell_lighting",
    duration_ms=frame_duration_ms)

Square-root-scaled flow streaks trace hole collection alongside the illuminated current-voltage curve. As light turns on at short circuit, photogenerated holes rise out of the base, including laterally into the junction above shaded columns, and stream along the thin emitter into the front fingers while delivered current climbs proportionally. As forward bias increases, delivered current stays nearly flat across the plateau until approaching roughly 0.68 V, where forward diode injection across the junction cancels collection while internal hole circulation continues unabated.

## Finger spacing sweep

Logarithmically spaced periods widen the gap from 2.5 finger widths to three base depths while tracking the maximum-power point via three-point parabolic interpolation from each spacing to the next. Running the cells below takes about twenty seconds.

In [ ]:
def power_peak(sample_biases: Scalar, sample_powers: Scalar) -> tuple[int, Scalar]:
    """Three-point parabolic interpolation around the highest interior sample of `sample_powers` (`[...] Scalar+`)
    locates the peak index and bias (`[] Scalar+`)."""
    best = int(sample_powers[1:-1].argmax()) + 1
    low, middle, high = sample_powers[best - 1], sample_powers[best], sample_powers[best + 1]
    step = (sample_biases[best + 1] - sample_biases[best - 1]) / 2
    return best, sample_biases[best] + step * ((low - high) / (2 * (low - 2 * middle + high))).clip(-1.5, 1.5)


bias_step = 1.5
lit_readings = [terminal_readings(areas, gradient, state) for state in lit_states]
lit_powers = stack([voltage * current for voltage, current in lit_readings]) / finger_spacings[0]   # [n_frames] Scalar+
best_frame, best_bias = power_peak(bias_extensor, lit_powers)
unshaded_power = lit_powers[best_frame] * finger_spacings[0] / (finger_spacings[0] - scenarios.finger_width)   # [] Scalar+

In [ ]:
def finger_sweep(potentials: Potentials, bias: Scalar) -> Iterator[tuple[Point, Potentials]]:
    """Yields `(vertices, potentials)` (`Point-[V]`, `Potentials+[V]`) near maximum power across `finger_spacings`."""
    for finger_spacing in finger_spacings:
        current_vertices = (mv(Upright, scenarios.cell_places(finger_spacing)) + mv.w).dual().field()   # Point-[V]
        step_areas, step_dual_areas, step_gradient = simplex_maps(current_vertices)
        trial_biases = bias + bias_step * np.array([-1.0, 0.0, 1.0])          # [3] Scalar+
        voltages = trial_biases[:, None] * finger_mask                        # [3, V_held] Scalar+
        trial_contacts = held_reading * neutral + (mv.t * voltages).field()   # [3] Potentials+[V_held]
        states = list(steady_states(step_areas, step_dual_areas, step_gradient, potentials, trial_contacts,
                                    np.ones(3) * generation))
        readings = [terminal_readings(step_areas, step_gradient, state) for state in states]
        _, bias = power_peak(trial_biases, stack([voltage * current for voltage, current in readings]))
        potentials = states[1]
        yield current_vertices, potentials

In [ ]:
def hole_flows_at(current_vertices: Point, potentials: Potentials) -> Point:
    """Simplex gradients at `current_vertices` (`Point-[V]`) determine the triangle hole flows (`Point-[F]`)."""
    return hole_flows(simplex_maps(current_vertices)[2], potentials)


def power_at(finger_spacing: float, current_vertices: Point, potentials: Potentials) -> Scalar:
    """Terminal voltage times delivered current per `finger_spacing` gives areal power density (`[] Scalar+`)."""
    step_areas, _, step_gradient = simplex_maps(current_vertices)
    voltage, current = terminal_readings(step_areas, step_gradient, potentials)
    return voltage * current / finger_spacing


spacing_states = show_states(
    finger_sweep(lit_states[best_frame], best_bias), render.finger_sweep_animation, topology, dopants, contacts,
    generation, hole_flows_at, power_at, reference_flow, finger_spacings, unshaded_power,
    name="solar_cell_spacing", duration_ms=frame_duration_ms)

Closely spaced fingers block a large fraction of incident sunlight while collecting emitter holes over a short lateral path. Widening the gap reduces optical shading and raises delivered power until lateral hole crowding along the thin emitter builds a resistive voltage drop from gap centre to finger edge that outweighs the extra light, leaving the open ring at the optimum spacing while wide-gap base streaks fan diagonally toward the fingers.

## Optimal finger spacing

Plotting delivered power per unit cell width across all spacings isolates the trade-off between contact shading and emitter sheet resistance and selects the optimal period.

In [ ]:
powers = stack([power_at(finger_spacing, *state)
                for finger_spacing, state in zip(finger_spacings, spacing_states)])   # [n_spacings] Scalar+
optimum = int(powers.argmax())
render.power_figure(finger_spacings, powers);
print(f"optimal finger spacing: {finger_spacings[optimum] / scenarios.finger_width:.1f} finger widths")

Delivered power per unit surface width rises smoothly as shading recedes, peaks near the marked ring at roughly thirteen finger widths, and falls off once lateral emitter resistance dominates.

In [ ]:
optimal_vertices, optimal_potentials = spacing_states[optimum]
render.period_figure(topology, optimal_vertices, dopants, contacts, generation,
                   hole_flows_at(optimal_vertices, optimal_potentials), reference_flow, finger_spacings[optimum]);

Across one period centred on a finger at the optimal spacing, holes generated across both adjacent half-gaps rise vertically into the emitter and converge symmetrically onto the central finger while base generation beneath the finger's shadow stays supplied by lateral diffusion.

In [ ]:
# checks
# Shaded dual cells beneath the opaque fingers have zero optical generation.
shadow_reading = topology.reading(np.flatnonzero(shadow_mask), 0)             # Scalar+[V_shaded <- V]
assert ((shadow_reading * (dual_areas * generation)).sites.sum().abs() < 1e-12).all()
# Widening the finger spacing monotonically increases photogenerated pairs per unit surface width.
made_per_length = stack([(simplex_maps(step_vertices)[1] * generation).sites.sum() / finger_spacing
                         for finger_spacing, (step_vertices, _) in zip(finger_spacings, spacing_states)])   # [n_spacings] Scalar+
assert (made_per_length[1:] > made_per_length[:-1]).all()
# Widening the finger spacing monotonically increases the lateral hole quasi-Fermi drop along the emitter surface.
surface = np.flatnonzero(coordinates[:, 1] == 0)
edge = surface[len(fingers) // 2 - 1:len(fingers) // 2]
middle = surface[len(surface) // 2:len(surface) // 2 + 1]
across_gap = topology.reading(middle, 0) - topology.reading(edge, 0)          # Scalar+[1 <- V]
lateral_drops = stack([(across_gap * (potentials & hole)).sites.sum()
                       for _, potentials in spacing_states]).abs()            # [n_spacings] Scalar+
assert bool((lateral_drops[1:] > lateral_drops[:-1]).all()) and bool(lateral_drops[-1] > 30 * lateral_drops[0])
# Delivered power peaks strictly inside the sweep more than 10% above both endpoints.
assert 0 < optimum < n_spacings - 1 and 10 < finger_spacings[optimum] / scenarios.finger_width < 20
assert bool(powers[0] < 0.9 * powers[optimum]) and bool(powers[-1] < 0.9 * powers[optimum])
# At open circuit, optical absorption conserves parcel energy and produces strictly positive entropy everywhere.
sun = 5800 / 300
absorption = pairs - gap * light                                              # [] Point-
open_circuit = lit_states[-1]                                                 # Potentials+[V]
energies = (open_circuit & time) * mv.t + gap / 2 * (mv.n + mv.p) + mv.q + mv.l   # Plane+[V]
entropy_slope = energies - open_circuit - (1 - 1 / sun) * mv.l                # Plane+[V]
assert ((energies & absorption).batch().abs() < 1e-10).all()
assert ((entropy_slope & absorption).batch() > 0).all()